# scRNA-seq Doublet Detection — Consensus via Rank Aggregation

This notebook documents how we detect and summarize **doublets** in scRNA-seq by
(1) running one or more **doublet score algorithms** per sublib/flowcell, then  
(2) combining their outputs with **rank aggregation** to form a **consensus**.

It is written to be readable in an **HTML export**: each figure is followed by a short caption and practical interpretation.

---

## What this notebook does

1. **Collects per-method outputs** (scores and/or calls) produced by the workflow.
2. **Converts scores to ranks per method** (rank 1 = strongest doublet evidence).
3. **Aggregates ranks across methods** (median, rank-product, Borda, RRA).
4. **Shows concordance** (heatmap + Upset plots) and **diagnostic views** (histograms, pairplots).
5. **Summarizes decisions** (e.g., majority vote / threshold rules) for downstream filtering.

> **Rank direction.** Throughout this report, **lower rank numbers indicate *more doublet-like*** (i.e., rank 1 is the “worst” singlet candidate). When p-values are shown (e.g., RRA), **smaller p** = stronger aggregate evidence for doublet.

---

## Doublet score algorithms (per-cell scores/calls)

These are the **base detectors**. The workflow may include any subset of the following (depending on your config):

- **Scrublet** (Python): Simulates doublets and scores cells; widely used baseline.
- **scDblFinder** (R/Bioconductor): Model-based detection with robust defaults; outputs scores/calls.
- **SCDS family** (R):  
  - **cxds** (co-expression) and **bcds** (binary), often used as **hybrid**; provide doublet scores.
- **Solo** (deep learning): VAE-based; produces probabilities/calls (e.g., `is_doublet.npy`).
- **SoCube** (deep learning): CNN/transformer-style scoring over gene expression matrices.

**How these feed the notebook:** Each method yields either a **score** (higher = more doublet-like) or a **binary call**.  
Scores are **ranked within method** so methods become comparable; calls are summarized separately (e.g., majority vote).

---

## Rank aggregation methods (combine methods into a consensus)

These operate **on ranks**, not raw scores:

- **Median rank** — robust to a single outlier method; conservative if methods disagree widely.
- **Rank-product** — geometric mean of ranks; highlights cells **consistently high-ranked** across methods.
- **Borda count** — sum of ranks; simple, gives each method equal weight.
- **RRA (Robust Rank Aggregation)** — statistical test asking whether a cell’s ranks are **unexpectedly good** across methods vs. random; yields a **p-value** (smaller = stronger evidence).

**Consensus call schemes (binary):**
- **Majority vote** over per-method calls (ties → unassigned).  
- Optional **p-value thresholds** for RRA / rank-product can be used to mark high-confidence doublets.

---

## How to read the figures

- **Global/Per-method histograms of ranks/scores**  
  Right-skewed tails typically correspond to **putative doublets**. Clear separation between groups is good.

- **Composite overview (heatmap + Upset plots)**  
  - **Left heatmap**: per-cell ranks across methods (rows are cells, columns = methods).  
    Block patterns indicate **consistent evidence** across methods.  
  - **Top-right Upset**: which **methods agree a cell is singlet**.  
  - **Bottom-right Upset**: agreement for the **majority-vote** decision.

- **Pairplots (raw scores vs. aggregate ranks)**  
  Smooth gradients/monotone trends show **agreement** between raw method scores and the **consensus ranks**; outliers flag disagreements worth review.

- **Per-sample summaries** (if present)  
  Doublet rates by sample; outliers can reflect **overloading**, chemistry differences, or ambient RNA.

---

## Interpretation guide

- **Lower rank number = stronger doublet evidence.**  
- **Consistency across methods > any single method.**  
- Use **RRA p-values** (small = strong) or **rank-product** to prioritize high-confidence doublets.  
- Review **sample outliers** and clusters enriched for doublets on embeddings before filtering.

---

*Provenance:* This notebook consumes outputs from the project’s doublet rules (e.g., Scrublet, scDblFinder, SCDS [cxds/bcds/hybrid], Solo, SoCube) and combines them via median, rank-product, Borda, and RRA. Majority vote is used for a simple binary consensus when needed.


In [ ]:
import sys
import os
import warnings
warnings.filterwarnings(action="ignore")

import numpy as np
import pandas as pd
from scipy.stats import rankdata, beta
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import marsilea as ma
from marsilea.plotter import ColorMesh, Colors, FixedChunk, Chunk, Labels,Title
from marsilea.upset import UpsetData, Upset

In [ ]:
try:
    import mpl_fontkit as fk
    fk.install("Lato")
except:
    pass

In [ ]:
# --- Notebook/HTML narration helpers ---------------------------------------
try:
    from IPython.display import display, Markdown as _MD
except Exception:  # not in a notebook
    _MD = None

def _md(text: str):
    """Render Markdown in notebooks; fallback to print in scripts."""
    if _MD is not None:
        display(_MD(text))
    else:
        print(text)

def _bold(s: str) -> str:
    return f"**{s}**"

# Auto-numbered figure captions
import itertools as _it
_FIGNO = _it.count(1)
def caption(text: str, anchor: str | None = None):
    n = next(_FIGNO)
    anchor = anchor or f"fig{n}"
    _md(f"<a id='{anchor}'></a>{_bold(f'Figure {n}.')} {text.strip()}")

In [ ]:
def read_score_call_files(input_files, join="inner", rank_ascending=True,
                            rank_na_option="bottom", score_column="doublet_score",
                            call_column="doublet", parent_dir_level=1):
    """
    Read tab-separated files and extract specified score and call columns, supporting files with differing indices.

    Parameters
    ----------
    input_files : list of str
        List of file paths to the input files. Each file is expected to be a tab-separated
        file with an index in the first column and to contain at least the score column.
    join : str, optional
        The type of join to perform when concatenating indices from different files.
        The default is 'inner', which keeps only the indices common to all files. Use 'outer'
        to take the union of all indices.
    rank_ascending : bool, optional
        Whether the ranking should be in ascending order. The default is True.
    rank_na_option : str, optional
        How to handle NA values during ranking. Options include 'keep', 'top', or 'bottom'.
        The default is 'bottom'.
    score_column : str, optional
        The name of the column containing the scores. The default is "doublet_score".
    call_column : str or None, optional
        The name of the column containing the calls. The default is "doublet". If set to None,
        calls will not be processed and the returned calls_df will be None.
    parent_dir_level : int, optional
        The level of the parent directory to be used as the column name.
        A value of 1 means the immediate parent directory (default), 2 means the parent's parent, etc.

    Returns
    -------
    rank_df : pandas.DataFrame
        DataFrame containing the ranked scores, where each column corresponds to a method
        (extracted from the file path based on parent_dir_level). Missing values are represented as NaN.
    calls_df : pandas.DataFrame or None
        DataFrame containing the calls (converted to categorical data), where each column corresponds
        to a method (extracted from the file path based on parent_dir_level). Missing values are represented
        as 'unassigned'. If call_column is None, this will be None.

    Notes
    -----
    - The method name is extracted by splitting the file path using the OS separator and taking the element
      at position -(parent_dir_level + 1). If the file path does not have enough levels, a fallback is used.
    - If files have different indices, the resulting DataFrames will have indices based on the join type specified.
    """
    scores = {}
    calls = {}

    for fn in input_files:
        try:
            df = pd.read_table(fn, index_col=0)
        except Exception as e:
            print("Error reading file {}: {}".format(fn, e))
            continue

        # Extract the method name based on the specified parent directory level.
        parts = os.path.normpath(fn).split(os.sep)
        try:
            method = parts[-(parent_dir_level + 1)]
        except IndexError:
            print("Warning: file {} does not have enough directory levels; using default method name.".format(fn))
            method = os.path.basename(os.path.dirname(fn))

        # Check if the score column exists.
        if score_column not in df.columns:
            print("File {} is missing the required score column '{}'.".format(fn, score_column))
            continue

        scores[method] = df[score_column]

        # Process call column only if provided.
        if call_column is not None:
            if call_column not in df.columns:
                print("File {} is missing the required call column '{}'.".format(fn, call_column))
                continue
            calls[method] = df[call_column]

    # Concatenate the score series into a DataFrame.
    scores_df = pd.concat(scores, axis=1, join=join)
    # Rank the scores DataFrame using the specified rank_na_option.
    rank_df = scores_df.rank(axis=0, na_option=rank_na_option, ascending=rank_ascending)

    # Process calls only if call_column is provided and any calls were collected.
    calls_df = None
    if call_column is not None and calls:
        calls_df = pd.concat(calls, axis=1, join=join)
        calls_df = calls_df.fillna("unassigned").astype("category")

    return rank_df, calls_df, scores_df


In [ ]:
def majority_vote(df, add_key=None, split_vote_name="unassigned"):
    """
    Determine the majority vote for each row of a DataFrame.

    This function calculates the mode (most frequent value) along each row.
    For each row, if a unique mode exists (i.e., the other mode columns are NaN),
    that value is used as the majority vote. If multiple modes exist (i.e., a tie),
    the vote is set to the specified split_vote_name.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing vote data. Each row represents a set of votes.
    add_key : str or None, optional
        If provided, the computed majority vote will be added as a new column with this name
        to the input DataFrame. If None, only the majority vote Series is returned.
    split_vote_name : str, optional
        The value to assign to rows with tied votes (i.e., no unique majority). The default is
        "unassigned".

    Returns
    -------
    majority : pandas.Series or pandas.DataFrame
        If add_key is None, returns a pandas Series with the majority vote for each row.
        Otherwise, returns the original DataFrame with a new column (named add_key) added,
        containing the majority vote.
    
    Notes
    -----
    - The mode is computed row-wise using pandas.DataFrame.mode.
    - When multiple mode candidates are returned for a row, the function checks if any of the
      mode columns is NaN. A NaN indicates that there was only one valid mode and the other columns
      are merely padding; in that case, the unique mode is used. If no NaN is present, it is treated
      as a tie, and the split_vote_name is used.
    - The function modifies the input DataFrame in-place if add_key is provided.
    """
    # Compute the mode for each row.
    mode_df = df.mode(axis=1)

    # If more than one candidate exists, check for ties.
    if mode_df.shape[1] > 1:
        # For rows with a unique mode, the extra columns are NaN.
        # If any column in a row is NaN, use the first column as the unique mode;
        # otherwise, assign split_vote_name to indicate a tie.
        majority = np.where(mode_df.isna().any(axis=1),
                            mode_df.iloc[:, 0],
                            split_vote_name)
    else:
        # If there's only one mode candidate per row, use it directly.
        majority = mode_df.iloc[:, 0]

    # Ensure the result is a pandas Series with the same index as df.
    majority = pd.Series(majority, index=df.index)

    if add_key is None:
        return majority
    else:
        df[add_key] = majority
        return df


In [ ]:
def _rho_scores(rmat, dist_a, dist_b, epsilon=1e-300):
    """
    Calculate Beta Distribution Rho Scores and compute a visualization score.

    This function sorts the input rank matrix row-wise, computes the Beta cumulative 
    distribution function (CDF) for each element using the provided shape parameters, 
    extracts the minimum p-value per row, and then applies a correction by multiplying 
    by the number of methods and clipping the result to 1 (rho score). In addition, it 
    computes a visualization score as -log10(raw p-value), where raw p-values are the 
    minimum p-values before correction.

    Parameters
    ----------
    rmat : array_like
        A matrix (or DataFrame) where each row contains normalized rank values for an 
        interaction across different methods.
    dist_a : array_like
        Array of shape parameters (a) for the Beta distribution.
    dist_b : array_like
        Array of shape parameters (b) for the Beta distribution.
    epsilon : float, optional
        A small constant to avoid taking the logarithm of zero. Default is 1e-300.

    Returns
    -------
    rho : numpy.ndarray
        A 1D array of corrected p-values (rho scores) for each interaction.
    score : numpy.ndarray
        A 1D array of visualization scores computed as -log10(raw p-value), where raw 
        p-values are the minimum p-values before correction.
    """
    # Sort the rank matrix row-wise.
    rmat_sorted = np.sort(rmat, axis=1)
    
    # Compute the Beta CDF for each element.
    p_values = beta.cdf(rmat_sorted, dist_a, dist_b)
    
    # Extract the minimum p-value for each row.
    raw_p = np.min(p_values, axis=1)
    
    # Correct the p-values by multiplying by the number of methods (columns)
    # and clipping to the range [0, 1].
    rho = np.clip(raw_p * rmat.shape[1], a_min=0, a_max=1)
    
    # Compute visualization score: replace zero raw p-values with epsilon 
    # to avoid -log10(0), then compute -log10(raw p-value).
    raw_p_adj = np.where(raw_p == 0, epsilon, raw_p)
    score = -np.log10(raw_p_adj)
    
    return rho, score

def robust_rank_aggregate(rmat):
    """
    Calculate Robust Rank Aggregate (RRA) p-values and visualization scores 
    as described in Kolde et al., 2012.

    The function normalizes the input rank matrix to the [0, 1] interval by dividing 
    each element by the maximum rank in its respective column. It then generates the 
    shape parameters for the Beta distribution based on the normalized ranks and calculates 
    the aggregated p-values and visualization scores using the _rho_scores function.

    If rmat is a pandas DataFrame, the outputs rho and score will be returned as 
    pandas Series with the same index as rmat.

    Parameters
    ----------
    rmat : array_like or pandas.DataFrame
        A matrix of interaction ranks for each method (columns). Each row corresponds 
        to an interaction and contains rank values for that interaction across different methods.

    Returns
    -------
    rho : numpy.ndarray or pandas.Series
        A 1D array/Series of corrected p-values for each interaction, representing the aggregated 
        significance of the ranks according to the RRA method.
    score : numpy.ndarray or pandas.Series
        A 1D array/Series of visualization scores computed as -log10(raw p-value) for each interaction.

    Notes
    -----
    This function is a ChatGPT-modified version of code originally from liana-py:
    https://github.com/saezlab/liana-py
    """
    # Preserve the index if rmat is a DataFrame.
    index = rmat.index if isinstance(rmat, pd.DataFrame) else None

    # Normalize the rank matrix to the [0, 1] interval.
    rmat_norm = rmat / np.max(rmat, axis=0)
    
    # Determine the number of methods (columns).
    n_methods = rmat.shape[1]
    
    # Generate shape parameters for the Beta distribution:
    # dist_a increments from 1 to n_methods for each interaction.
    dist_a = np.tile(np.arange(1, n_methods + 1), (rmat.shape[0], 1))
    # dist_b is defined as: n_methods - dist_a + 1.
    dist_b = n_methods - dist_a + 1
    
    # Calculate aggregated p-values and visualization scores.
    rho, score = _rho_scores(rmat_norm, dist_a, dist_b)
    
    # If the input was a DataFrame, convert outputs to Series with the same index.
    if index is not None:
        rho = pd.Series(rho, index=index)
        score = pd.Series(score, index=index)
    
    return rho, score

def borda_count_aggregate(rmat):
    """
    Aggregate ranks using the Borda count method.

    For each interaction (row), this function computes the sum of ranks across methods.
    Lower total scores indicate better aggregated rankings.

    Parameters
    ----------
    rmat : array_like or pandas.DataFrame
        A matrix of interaction ranks for each method (columns).

    Returns
    -------
    aggregated : numpy.ndarray or pandas.Series
        The aggregated Borda scores for each interaction (lower is better).
    """
    if isinstance(rmat, pd.DataFrame):
        aggregated = rmat.sum(axis=1)
        return aggregated
    else:
        return np.sum(rmat, axis=1)

def median_rank_aggregate(rmat):
    """
    Aggregate ranks by computing the median rank for each interaction.

    Parameters
    ----------
    rmat : array_like or pandas.DataFrame
        A matrix of interaction ranks for each method (columns).

    Returns
    -------
    median_rank : numpy.ndarray or pandas.Series
        The median rank for each interaction.
    """
    if isinstance(rmat, pd.DataFrame):
        median_rank = rmat.median(axis=1)
        return median_rank
    else:
        return np.median(rmat, axis=1)

def rank_product_aggregate(rmat, epsilon=1e-10):
    """
    Aggregate ranks using the rank product method.

    The rank product is computed as the geometric mean of the ranks for each interaction.
    Lower rank products indicate a higher overall ranking.

    Parameters
    ----------
    rmat : array_like or pandas.DataFrame
        A matrix of interaction ranks for each method (columns).
    epsilon : float, optional
        A small constant added to avoid issues with zero ranks. Default is 1e-10.

    Returns
    -------
    rank_product : numpy.ndarray or pandas.Series
        The aggregated rank product for each interaction (lower is better).
    """
    if isinstance(rmat, pd.DataFrame):
        n_methods = rmat.shape[1]
        # Add epsilon to avoid zero multiplication issues
        product = (rmat + epsilon).prod(axis=1)
        rank_prod = product ** (1.0 / n_methods)
        return rank_prod
    else:
        n_methods = rmat.shape[1]
        product = np.prod(rmat + epsilon, axis=1)
        return product ** (1.0 / n_methods)

def rank_product_permutation(rmat, n_permutations=1000, epsilon=1e-10):
    """
    Perform a permutation analysis for the rank product method.

    This function computes the observed rank product for each interaction (row) using the rank_product_aggregate function.
    Then, it generates a null distribution by independently shuffling the values in each column a number of times.
    For each row, the permutation-based p-value is computed as the fraction of permutations where the permuted rank product
    is less than or equal to the observed rank product (assuming lower values indicate a stronger signal).

    Parameters
    ----------
    rmat : array_like or pandas.DataFrame
        A matrix of interaction ranks for each method (columns). Each row corresponds to an interaction.
    n_permutations : int, optional
        The number of permutations to perform. Default is 1000.
    epsilon : float, optional
        A small constant added to avoid issues with zero ranks. Default is 1e-10.

    Returns
    -------
    observed_rp : pandas.Series or numpy.ndarray
        The observed rank product for each interaction.
    p_values : pandas.Series or numpy.ndarray
        The permutation-based p-value for each interaction, computed as the fraction of permutations
        where the permuted rank product is less than or equal to the observed rank product.

    Notes
    -----
    This function assumes that lower rank product values indicate a stronger signal.
    If the input is a pandas DataFrame, the returned observed_rp and p_values are pandas Series with the same index.
    
    Example
    -------
    >>> observed, pvals = rank_product_permutation(rmat, n_permutations=1000)
    """
    # Preserve index if rmat is a DataFrame.
    if isinstance(rmat, pd.DataFrame):
        index = rmat.index
        data = rmat.values.copy()
    else:
        index = None
        data = np.array(rmat)

    n_rows, n_methods = data.shape

    # Compute the observed rank product.
    observed_rp = rank_product_aggregate(rmat, epsilon=epsilon)
    
    # Initialize a counter for each row.
    counts = np.zeros(n_rows)

    # Permutation loop: shuffle each column independently.
    for i in range(n_permutations):
        permuted = np.empty_like(data)
        for j in range(n_methods):
            permuted[:, j] = np.random.permutation(data[:, j])
        
        # Compute rank product for the permuted data.
        permuted_rp = rank_product_aggregate(permuted, epsilon=epsilon)
        # Increase count for rows where permuted rank product is less than or equal to the observed.
        counts += (permuted_rp <= observed_rp).astype(int)
    
    # Compute empirical p-values using the formula: (counts + 1) / (n_permutations + 1)
    p_values = (counts + 1) / (n_permutations + 1)
    # If input was a DataFrame, convert results to pandas Series.
    if index is not None:
        observed_rp = pd.Series(observed_rp, index=index)
        p_values = pd.Series(p_values, index=index)
    
    return observed_rp, p_values

In [ ]:
def rank_aggregation_and_majority_vote(rmat, calls, n_permutations=10000, rra_lim=0.01, rp_lim=0.05, plot=True):
    vote = majority_vote(calls, add_key="majority_vote")
    rra_pval, rra_score = robust_rank_aggregate(rmat)
    rp, rp_pval = rank_product_permutation(rmat, n_permutations=n_permutations)
    
    rmat["rp_score"] = pd.Series(rp, index=rmat.index)
    rmat["rp_rank"] = rp.rank(ascending=True, na_option="bottom")
    rmat["rp_pval"] = pd.Series(rp_pval, index=rmat.index)
    rmat["rra_score"] = pd.Series(rra_score, index=rmat.index)
    rmat["rra_rank"] = rra_score.rank(ascending=False, na_option="bottom")
    rmat["rra_pval"] = pd.Series(rra_pval,  index=rmat.index)
    
    vote["rra"] = ["doublet" if x <= rra_lim else "singlet" for x in rra_pval]
    vote["rp"] = ["doublet" if x <= rp_lim else "singlet" for x in rp_pval]
    
    if plot:
        med_rank = median_rank_aggregate(rmat)
        borda = borda_count_aggregate(rmat) / rmat.shape[1]
        res = pd.concat([med_rank, rp, borda, rra_score.rank(ascending=False), vote['majority_vote']], axis=1)
        res.columns = ['med_rank', 'rp', 'borda', 'rra', 'majority_vote']
        res_m = res.melt(id_vars=['majority_vote'], var_name="method", value_name="rank")
        sns.displot(data=res_m, kind='hist', col="method", col_wrap=2, x="rank", hue="majority_vote", legend=True, facet_kws={'sharex': False, 'sharey': False})
        caption("Distribution of aggregate ranks by method (median, rank-product, Borda, RRA) "
                "split by majority-vote class. "
                "Separation between singlets and doublets across panels indicates consistent detection.",
                anchor="fig-agg-hists")
        

    return rmat, vote

In [ ]:
def get_set_colors(num_sets):
    """Returns a dictionary of fixed colors for Sets of size 2, 3, or 4."""
    if num_sets == 2:
        return {
            "10": "#1f77b4",  # Set A (Blue)
            "01": "#ff7f0e",  # Set B (Orange)
            "11": "#b5651d",  # A & B (Brown)
        }
    elif num_sets == 3:
        return {
            "100": "#1f77b4",  # Set A (Blue)
            "010": "#ff7f0e",  # Set B (Orange)
            "001": "#2ca02c",  # Set C (Green)
            "110": "#b5651d",  # A & B (Brown)
            "101": "#17becf",  # A & C (Teal)
            "011": "#9c8704",  # B & C (Olive)
            "111": "#7e5a9b",  # A & B & C (Muted Purple)
        }
    elif num_sets == 4:
        return {
            "1000": "#1f77b4",  # Set A (Blue)
            "0100": "#ff7f0e",  # Set B (Orange)
            "0010": "#2ca02c",  # Set C (Green)
            "0001": "#d62728",  # Set D (Red)
            "1100": "#b5651d",  # A & B (Brown)
            "1010": "#17becf",  # A & C (Teal)
            "1001": "#9467bd",  # A & D (Purple)
            "0110": "#9c8704",  # B & C (Olive)
            "0101": "#e377c2",  # B & D (Pink)
            "0011": "#8c564b",  # C & D (Dark Brown)
            "1110": "#7f7f7f",  # A & B & C (Gray)
            "1101": "#bcbd22",  # A & B & D (Yellow-Green)
            "1011": "#1f77b4",  # A & C & D (Bluish)
            "0111": "#ff9896",  # B & C & D (Pale Red)
            "1111": "#000000",  # A & B & C & D (Black)
        }
    else:
        raise ValueError("Only supports Sets with 2, 3, or 4 sets.")


In [ ]:
import pandas as pd

def compute_set_format(data, set_names=None):
    """
    Computes set membership format strings ('100', '110', etc.) and human-readable labels ('A + B', etc.)
    from one of the following inputs:

    1. A dictionary of membership lists, where keys are set names and values are lists of items in that set.
    2. A binary indicator matrix (pandas DataFrame), where rows represent elements and columns represent sets.
    3. A categorical pandas Series, where index values are element names, and categories are the set names.
    4. A single-column pandas DataFrame that appears to contain categorical-like data.

    Parameters:
    - data (pd.DataFrame, dict of lists, or pd.Series): Input data.
      * If a DataFrame, rows represent elements, and columns represent sets (with binary values: 1 or 0).
      * If a dict, keys are set names and values are lists of elements belonging to that set.
      * If a Series, its categories represent the sets, and its index represents elements.
      * If a single-column DataFrame, it is inferred as a categorical Series if it has low cardinality.
    - set_names (list, optional): Column names for sets (only needed when `data` is a dict or Series).
      If not provided, the sorted unique categories (for Series) or sorted keys (for dict) will be used.

    Returns:
    - pd.DataFrame: A DataFrame with three columns:
        * 'format': A binary membership string (e.g., '101') representing membership across sets.
        * 'label' : A human-readable label (e.g., 'A + C') representing membership.
        * 'colors' : A color label for visualization (if `get_set_colors` is defined).
      The DataFrame's index corresponds to the element names.
    """
    # Check for empty input
    if data is None or (isinstance(data, (dict, pd.DataFrame, pd.Series)) and len(data) == 0):
        raise ValueError("Input data cannot be empty.")

    # If input is a single-column DataFrame, check if it resembles a categorical Series
    if isinstance(data, pd.DataFrame) and data.shape[1] == 1:
        column = data.iloc[:, 0]
        unique_values = column.nunique()
        total_values = len(column)

        # If the column has low cardinality, assume it represents categorical data
        if unique_values < min(30, total_values * 0.2) or pd.api.types.is_categorical_dtype(column):
            data = column.astype("category")

    # Handle categorical Series input
    if isinstance(data, pd.Series):
        # Ensure it's categorical
        if not pd.api.types.is_categorical_dtype(data):
            data = data.astype("category")

        # If set_names is not provided, use the unique categories
        if set_names is None:
            set_names = sorted(data.cat.categories)

        # Convert categorical Series to indicator matrix
        data = pd.get_dummies(data).reindex(columns=set_names, fill_value=0)

    # Convert dictionary input to an indicator DataFrame
    elif isinstance(data, dict):
        if set_names is None:
            set_names = sorted(data.keys())
        unique_items = sorted(set.union(*map(set, data.values())))
        data = pd.DataFrame(
            [[1 if item in data.get(set_name, []) else 0 for set_name in set_names] for item in unique_items],
            index=unique_items,
            columns=set_names
        )
    
    elif not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a dictionary of lists, a pandas DataFrame, or a categorical Series.")

    # Ensure set_names is assigned
    set_names = list(data.columns)

    # Fix: Explicitly cast values to int to avoid "TrueFalseTrue" issues
    format_strings = [''.join(map(str, row.astype(int))) for _, row in data.iterrows()]

    readable_labels = [
        ' + '.join([set_names[i] for i, bit in enumerate(fmt) if bit == '1'])
        if '1' in fmt else 'None'
        for fmt in format_strings
    ]

    # Ensure `get_set_colors` is defined and returns a color mapping
    try:
        lut = get_set_colors(data.shape[1])
        colors = [lut.get(fmt, 'gray') for fmt in format_strings]
    except NameError:
        colors = ['gray'] * len(format_strings)

    # Construct result DataFrame
    result_df = pd.DataFrame({"format": format_strings, "label": readable_labels, "colors": colors}, index=data.index)
    
    return result_df


In [ ]:
def _color_upset_subsets(cls, colors, cmap=None):
    """add color to set subset overlaps
    """
    for i, m in enumerate(cls.data.mark()):
        cls._subset_styles[i] = dict(facecolor=colors[i], edgecolor=colors[i])
        cls._subset_line_styles[i] = dict(color=colors[i])

def _rank_heatmap(rmat, sets, vote, keep=None, cmap="YlGnBu", label="Rank", clust_metric="correlation", group=False):
    if keep is not None:
        rmat = rmat.loc[keep]
        sets = sets.loc[keep]
        vote = vote.loc[keep]
    rmat = rmat.drop(columns=['rp_score', 'rp_pval', 'rra_score', 'rra_pval'])
    #h1 = ma.Heatmap(rmat.values, width=4, height=8, vmax=rmat.max().mean(), cmap="YlGnBu", label="Rank")
    h1 = ma.Heatmap(rmat.values, width=4, height=8, cmap="YlGnBu", label="Rank")
    h1.add_top(Labels(rmat.columns, label=None, label_loc="top"), pad=0.1) # top labels
    h1.add_dendrogram("top", show=True)
    cpal = sets.value_counts().reset_index()[["label", "colors"]].set_index("label")["colors"].to_dict()
    h1.add_left(Colors(sets["label"], palette=cpal, label="Singlet Calls", label_loc="top"), pad=0.1) # left colormesh
    if clust_metric == "kendall_tau":
        h1.add_dendrogram("left", add_base=False, linkage=linkage_matrix)
    else:
        h1.add_dendrogram("left", method="average", metric=clust_metric, add_meta=False)
    if group:
        h1.group_rows(vote["majority_vote"])
        row_group_labels = ["doublet", "singlet", "unassigned"] if "unassigned" in sets["label"] else ["doublet", "singlet"]
        h1.add_right(Chunk(row_group_labels, bordercolor="gray"), pad=0.1)
    h1.add_legends()
    return h1

def _call_upsetplot(vote, sets, call_type, keep=None):
    indmat = (vote[["majority_vote", "rra", "rp"]] == call_type).astype(int)
    if keep is not None:
        vote = vote.loc[keep]
        sets = sets.loc[keep]
    indmat = (vote[["majority_vote", "rra", "rp"]] == call_type).astype(int)
    dat = UpsetData(indmat)
    h2 = Upset(dat, orient="h", width=4, height=2, sets_color=["gray"] * indmat.shape[1])
    h2.add_top(Title("Singlet Calls", align="left", padding=20))
    cpal2 = sets.value_counts().reset_index()[["format", "colors"]].set_index("format")["colors"]
    dd = h2.sets_table.index.to_frame()[indmat.columns]
    format_strings = [''.join(map(str, row.astype(int))) for _, row in dd.iterrows()]
    colors = [cpal2[i] for i in format_strings]
    _color_upset_subsets(h2, colors)
    return h2

def _majorty_vote_upsetplot(vote, call_type, keep=None):
    indmat = (vote[vote_methods] == call_type).astype(int)
    if keep is not None:
        indmat = indmat.loc[keep,:]
    dat = UpsetData(indmat)
    h3 = Upset(dat, orient="h", width=4, height=2, sets_color=["gray"] * len(vote_methods), min_cardinality=100)
    h3.highlight_subsets(facecolor='#E50046', label="singlet", min_degree=3, max_degree=10)
    h3.add_top(Title("Majority Vote", align="left", padding=20))
    return h3
    
def create_figure(rmat, vote, sets, call_type="singlet", clust_metric="correlation", keep=None, group=True):
    h1 = _rank_heatmap(rmat, sets, vote, keep=keep, group=group)
    h2 = _call_upsetplot(vote, sets, call_type)
    h3 =  _majorty_vote_upsetplot(vote, call_type)
    
    sb1 = ma.StackBoard([h2, h3], direction="vertical", align="left", spacing=1)
    sb = ma.StackBoard([h1, sb1], direction="horizontal", align="top", spacing=0.5)
    sb.add_legends("top", stack_size=1, stack_by="column", align_stacks="top", stack_spacing=50)
    fig = plt.figure(figsize=(10, 10))
    fig.suptitle("Summary of rank aggregated doublet scores and voting on singlet calls", x=0.3, y=1.27, fontsize=16, fontweight="bold")
    #sb.render(figure=fig)
    return sb

In [ ]:
def save_data(vote, snakemake, fig=None, method="rra", rankdata=None):
    snakemake.params.method = snakemake.params.method or method
    if snakemake.params.method == "auto":
        if len(snakemake.input_files) % 2:
            snakemake.params.method = method
        else:
            snakemake.params.method = "majority_vote"
    if snakemake.params.method.lower() in vote.columns:
        output = vote[[snakemake.params.method.lower()]]
        output.columns = ["droplet_type"]
        output.to_csv(snakemake.output.combined, sep="\t")
    else:
        print(f"Invalid method: {snakemake.params.method}. Valid method params: {','.join(vote.columns)}.")
    if rankdata is not None:
        fn = snakemake.output.combined[:-4] + "_rankdata.tsv"
        rankdata.to_csv(fn, sep="\t")
    if fig:
        fig.save(snakemake.output.figure)

In [ ]:
rmat, calls, scores = read_score_call_files(snakemake.input)
vote_methods = rmat.columns
n_vote_methods = len(vote_methods)
call_type = "singlet"

In [ ]:
_md("## Doublet rank aggregation — run metadata")
_md(f"- {_bold('Methods included')}: {', '.join(map(str, vote_methods))}")
_md(f"- {_bold('Join on barcodes')}: inner (shared only)")
_md(f"- {_bold('Call type shown in Upset plots')}: {call_type}")
_md(f"- {_bold('Decision rules')}: "
    "RRA p≤0.01 → doublet; Rank-Product p≤0.05 → doublet; "
    "majority vote across methods; ties → 'unassigned'.")


In [ ]:
# Special case when we have only one doublet score method
if n_vote_methods == 1:
    dd = scores.merge(calls, left_index=True, right_index=True)
    dd.columns = ['rank', "droplet_type"]
    p = sns.histplot(dd, x="rank", hue="droplet_type")
    calls.to_csv(snakemake.output.combined, sep="\t")
    p.figure.savefig(snakemake.output.figure, bbox_inches="tight")
    caption(
    "Histogram of the method’s doublet *score ranks* split by call. "
    "Right-skew / heavier tails typically indicate putative doublets; "
    "this plot documents the separation achieved by the single available method.",
    anchor="fig-single-hist")

    exit_notebook = True
else:
    exit_notebook = False
    
    

In [ ]:
if exit_notebook == False:
    rmat, vote = rank_aggregation_and_majority_vote(rmat, calls, n_permutations=10000, plot=True)
    plt.show()
    set_format = compute_set_format(vote[["majority_vote", "rra", "rp"]]==call_type)
    keep = (vote[["majority_vote", "rra", "rp"]] == "doublet").sum(1) > 0 # remove all barcodes called singlet in all samples
    barcodes_for_plot = keep[keep].index.to_list()
    sb = create_figure(rmat, vote, set_format, call_type=call_type, keep=barcodes_for_plot, group=True)
    sb.render()
    caption(
    "Composite summary: **left** heatmap shows per-barcode *ranks* across methods "
    "(rows clustered; left color bar encodes set membership of singlet calls). "
    "**Top-right** Upset plot: which methods *agree a cell is a singlet*; "
    "**bottom-right** shows the same for the *majority-vote* across all methods. "
    "Together these panels reveal method concordance and the barcode clusters that drive it.",
    anchor="fig-composite"
)

    save_data(vote, snakemake, fig=sb, rankdata=rmat)

In [ ]:
rmat.head()

In [ ]:
_scores = scores.merge(rmat[["rra_rank"]], left_index=True, right_index=True)
if exit_notebook == False:
    #sns.pairplot(_scores, hue="rra_rank", palette="YlGnBu", plot_kws=dict(linewidth=0, ), diag_kind="kde")
    sns.pairplot(_scores, plot_kws=dict(linewidth=0, s=10, alpha=0.5), palette="YlGnBu", diag_kind=None, hue="rra_rank")
    caption(
    "Pairwise scatter of raw doublet scores colored by **RRA rank** "
    "(lower rank = stronger aggregate signal). "
    "Look for monotone relationships and outliers: cohesive gradients "
    "reflect agreement with the aggregated ranking.",
    anchor="fig-pairplot"
)
